Now that I've classified the videos into content categories, and also joined their match context, I am going to run a regression analysis, looking at the factors of fan
engagement across format, timing, and subject, including how it tracks
against the match calendar.

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/lafc_content.db')


In [2]:
query = '''
SELECT *
FROM classified_videos
'''

df = pd.read_sql(query, conn)
df

,video_id,channel_id,title,description,published_at,duration,view_count,like_count,comment_count,fetched_at,format_family,content_type_final,dur_min
0,J8KtvKKDsBI,UCnqj91wjXAT0bsmxF1fHWBA,Inside LAFC | Episode 211 - A Strong Start,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-21T08:27:48Z,PT49M46S,1129,88,4,2026-07-21T18:13:52.868219+00:00,show,inside_lafc,NaN
1,6IGiJLX6zIA,UCnqj91wjXAT0bsmxF1fHWBA,Sonny's goal from pitchside 🤳,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-21T05:15:15Z,PT17S,9169,901,20,2026-07-21T18:13:52.868219+00:00,match,goal_clip,NaN
2,dk5FTY2zHEI,UCnqj91wjXAT0bsmxF1fHWBA,Son Heung-Min | EVERY ANGLE of his derby goal ...,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-20T07:17:45Z,PT2M10S,10782,1373,100,2026-07-21T18:13:52.868219+00:00,match,goal_clip,NaN
3,WGLkpecuCyA,UCnqj91wjXAT0bsmxF1fHWBA,LAFC Weekly | Episode 15 | 2026,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-19T01:00:21Z,PT21M,2106,167,9,2026-07-21T18:13:52.868219+00:00,show,lafc_weekly,NaN
4,rjdbdSE3TNo,UCnqj91wjXAT0bsmxF1fHWBA,A Night To Remember | LAG vs LAFC,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-07-18T22:58:28Z,PT25S,2759,457,31,2026-07-21T18:13:52.868219+00:00,match,match_preview,0.416667
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3566,8yoRN5MvJP0,UCnqj91wjXAT0bsmxF1fHWBA,Somos LAFC,"Nuestra Ciudad, Nuestro Club, Nuestro Escudo. ...",2016-01-08T00:27:56Z,PT2M,10264,165,16,2026-07-21T18:13:52.868219+00:00,social,misc_social_clip,2.000000
3567,Emczin-vSFQ,UCnqj91wjXAT0bsmxF1fHWBA,WE ARE LAFC,"Our City, Our Club, Our Crest. We are LAFC.",2016-01-07T17:51:44Z,PT2M,146275,1160,191,2026-07-21T18:13:52.868219+00:00,match,highlights,2.000000
3568,a63ytKgBTAc,UCnqj91wjXAT0bsmxF1fHWBA,John Thorrington announcement on SportsCenter,"Check out SportsCenter, giving some airtime to...",2015-12-09T23:54:13Z,PT27S,2094,24,2,2026-07-21T18:13:52.868219+00:00,social,misc_social_clip,0.450000
3569,QJP0ITdmAeo,UCnqj91wjXAT0bsmxF1fHWBA,Building Together: LAFC Stadium Workshop,We asked our supporters to help us design our ...,2015-11-24T19:50:33Z,PT1M1S,6295,73,5,2026-07-21T18:13:52.868219+00:00,match,match_preview,1.016667


In [3]:
with open('../sql/classified_videos_vs_lafc_match_context.sql') as f:
    query = f.read()

df = pd.read_sql_query(query, conn)
df.columns

DatabaseError: Execution failed on sql '-- This sql query joins the 'classified_videos' and the 'LAFC match context' tables, looking for the latest match kickoff time relative
-- to the publish time of each video. 

-- Also added a days since match column for later analysis. 

SELECT
  classified_videos.video_id,
  classified_videos.title,
  classified_videos.description,
  classified_videos.published_at,
  classified_videos.duration,
  classified_videos.view_count,
  classified_videos.like_count,
  classified_videos.comment_count,
  classified_videos.format_family,
  classified_content_type_final,

  lafc_match_context.season,
  lafc_match_context.kickoff_utc,
  lafc_match_context.opponent,
  lafc_match_context.result,                 -- 'W' / 'D' / 'L' (from LAFC's perspective)
  lafc_match_context.home_away,              -- 'H' / 'A' (from LAFC's perspective)
  lafc_match_context.goals_for,
  lafc_match_context.goals_against,

  -- LAFC's form going INTO the match (result above NOT 